# Phase 1: Setup & Imports
In this phase, we set up our environment and import the required libraries for our audio processing, deep learning model building, and evaluation pipeline.
We also include the Google Colab mount code, as required.

In [ ]:
# Mount Google Drive at start
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except ImportError:
    print("Running locally. Google Drive mount skipped.")

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import IPython.display as ipd
import tensorflow as tf

# Set plotting theme to dark to match the Streamlit dashboard style
plt.style.use('dark_background')

print("TensorFlow Version:", tf.__version__)
print("Librosa Version:", librosa.__version__)

# Phase 2: Dataset Exploration
Here, we explore the training dataset, counting the files in the `real/` (Genuine, label 0) and `fake/` (Deepfake, label 1) subfolders to analyze the class distribution. We will also load and plot raw waveforms, and play back samples of real vs fake speech.

In [ ]:
train_dir = 'data/for-2seconds/training'
categories = ['real', 'fake']

# Count files per class
counts = {}
for cat in categories:
    cat_path = os.path.join(train_dir, cat)
    if os.path.exists(cat_path):
        counts[cat] = len([f for f in os.listdir(cat_path) if f.lower().endswith(('.wav', '.mp3', '.flac'))])
    else:
        counts[cat] = 0

print("Dataset file counts:")
for cat, count in counts.items():
    print(f"  {cat.capitalize()} files: {count}")

# Plot class distribution
plt.figure(figsize=(6, 4))
plt.bar(counts.keys(), counts.values(), color=['#00C9A7', '#ff7b72'], width=0.6)
plt.title("Class Distribution in Training Set", fontsize=14, color='#00C9A7')
plt.xlabel("Audio Class")
plt.ylabel("Number of Samples")
plt.grid(True, alpha=0.1)
plt.show()

In [ ]:
# Load sample audio files
real_files = [f for f in os.listdir(os.path.join(train_dir, 'real')) if f.lower().endswith(('.wav', '.mp3', '.flac'))]
fake_files = [f for f in os.listdir(os.path.join(train_dir, 'fake')) if f.lower().endswith(('.wav', '.mp3', '.flac'))]

real_sample = os.path.join(train_dir, 'real', real_files[0])
fake_sample = os.path.join(train_dir, 'fake', fake_files[0])

# Playback samples in notebook
print("Genuine human voice sample:")
ipd.display(ipd.Audio(real_sample))

print("AI-Generated deepfake sample:")
ipd.display(ipd.Audio(fake_sample))

In [ ]:
# Load and plot raw waveforms side by side
y_real, sr_real = librosa.load(real_sample, sr=16000, mono=True)
y_fake, sr_fake = librosa.load(fake_sample, sr=16000, mono=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

librosa.display.waveshow(y_real, sr=sr_real, ax=ax1, color='#00C9A7')
ax1.set_title("Genuine (Human) Audio Waveform", color='#00C9A7')
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("Amplitude")

librosa.display.waveshow(y_fake, sr=sr_fake, ax=ax2, color='#ff7b72')
ax2.set_title("Deepfake (AI-Generated) Audio Waveform", color='#ff7b72')
ax2.set_xlabel("Time (s)")

plt.tight_layout()
plt.show()

# Phase 3: Feature Extraction
We extract both **MFCC (Mel-frequency cepstral coefficients)** and **Mel Spectrogram** features. We plot heatmaps of these features for both Genuine and Deepfake classes to highlight visual and acoustic differences, then show the process of extracting and saving them as `.npy` files.

In [ ]:
# Plot MFCC and Mel Spectrogram heatmaps for a real and fake voice
def plot_features(y, sr, title, label_color):
    # Extract features
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    # MFCC Plot
    img1 = librosa.display.specshow(mfcc, x_axis='time', ax=ax1, cmap='viridis')
    ax1.set_title(f"MFCC Heatmap ({title} Voice)", color=label_color)
    ax1.set_xlabel("Time")
    ax1.set_ylabel("MFCC Index")
    fig.colorbar(img1, ax=ax1)
    
    # Mel Spectrogram Plot
    img2 = librosa.display.specshow(mel_db, x_axis='time', y_axis='mel', sr=sr, ax=ax2, cmap='magma')
    ax2.set_title(f"Mel Spectrogram Log Power ({title} Voice)", color=label_color)
    ax2.set_xlabel("Time")
    fig.colorbar(img2, ax=ax2, format='%+2.0f dB')
    
    plt.tight_layout()
    plt.show()

plot_features(y_real, sr_real, "Genuine", '#00C9A7')
plot_features(y_fake, sr_fake, "Deepfake", '#ff7b72')

In [ ]:
# Load feature extraction functions
from utils.feature_extraction import extract_features, load_dataset

# We display saved numpy files generated from our training pipeline script
print("Checking for pre-extracted training and testing feature files...")
features_dir = 'features'
if os.path.exists(features_dir):
    files = os.listdir(features_dir)
    print("Files found in features directory:", files)
    
    # Let's inspect test arrays if they exist
    x_test_path = os.path.join(features_dir, 'X_test.npy')
    y_test_path = os.path.join(features_dir, 'y_test.npy')
    
    if os.path.exists(x_test_path) and os.path.exists(y_test_path):
        X_test = np.load(x_test_path)
        y_test = np.load(y_test_path)
        print(f"Successfully loaded testing arrays.")
        print(f"X_test shape: {X_test.shape} (N_samples, height, width, channels)")
        print(f"y_test shape: {y_test.shape} (N_samples,)")
else:
    print("Run training pipeline first (python train_pipeline.py) to extract and cache features!")

# Phase 4: Data Augmentation
To make our model robust and generalize better, we apply raw audio-level data augmentation. 
Here we visualize the effects of all four augmentations (Pitch Shifting, Time Stretching, Gaussian Noise, and Time Masking) on a single waveform.

In [ ]:
# Demonstrate augmentations on raw audio
# 1. Pitch shift (±2 semitones)
y_pitch_shifted = librosa.effects.pitch_shift(y_real, sr=sr_real, n_steps=2)

# 2. Time stretch (1.1x speed stretch)
y_time_stretched = librosa.effects.time_stretch(y_real, rate=1.1)

# 3. Gaussian noise (std=0.003)
y_noisy = y_real + np.random.normal(0, 0.003, len(y_real))

# 4. Time masking (zeroing out a random segment, e.g. 15% in middle)
y_masked = y_real.copy()
mask_len = int(len(y_real) * 0.15)
start_idx = int(len(y_real) * 0.4)
y_masked[start_idx : start_idx + mask_len] = 0.0

# Plot all versions
fig, axs = plt.subplots(5, 1, figsize=(12, 10), sharex=True, sharey=True)

librosa.display.waveshow(y_real, sr=sr_real, ax=axs[0], color='#00C9A7')
axs[0].set_title("Original Audio", color='#00C9A7')

librosa.display.waveshow(y_pitch_shifted, sr=sr_real, ax=axs[1], color='#9b5de5')
axs[1].set_title("Pitch Shifted (+2 semitones)", color='#9b5de5')

librosa.display.waveshow(y_time_stretched, sr=sr_real, ax=axs[2], color='#00bbf9')
axs[2].set_title("Time Stretched (1.1x)", color='#00bbf9')

librosa.display.waveshow(y_noisy, sr=sr_real, ax=axs[3], color='#f15bb5')
axs[3].set_title("Gaussian Noise Added", color='#f15bb5')

librosa.display.waveshow(y_masked, sr=sr_real, ax=axs[4], color='#ff7b72')
axs[4].set_title("Time Masked (SpecAugment style)", color='#ff7b72')

plt.tight_layout()
plt.show()

# Phase 5: Model Building
We construct a hybrid CNN-BiLSTM architecture. CNN blocks capture spatial-frequency details from the spectrogram channels, while the Bidirectional LSTM layers process temporal patterns to catch unnatural voice fluctuations.

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, MaxPooling2D, Dropout, Reshape, Bidirectional, LSTM, Dense

def build_model(input_shape=(128, 128, 2)):
    inputs = Input(shape=input_shape)
    
    # CNN Block 1
    x = Conv2D(32, (3,3), activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.2)(x)
    
    # CNN Block 2
    x = Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)
    x = Dropout(0.2)(x)
    
    # CNN Block 3
    x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,4))(x)  # Compress frequency dimension more aggressively
    x = Dropout(0.3)(x)
    
    # Reshape for LSTM: treat remaining frequency bins as features per timestep
    # Shape after pooling: (Batch, 16, 8, 128)
    # Reshaping to (8 timesteps, 16 * 128 features) = (8, 2048)
    x = Reshape((8, -1))(x)
    
    # BiLSTM Block
    x = Bidirectional(LSTM(128, return_sequences=True))(x)
    x = Dropout(0.3)(x)
    x = Bidirectional(LSTM(64, return_sequences=False))(x)
    x = Dropout(0.3)(x)
    
    # Output Fully Connected layers
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

model = build_model()
model.summary()

# Count model parameters
trainable_count = np.sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
print(f"\nTotal Trainable Parameters: {trainable_count:,}")

# Phase 6: Training
We plot the training and validation loss/accuracy curves from the completed training run to verify smooth convergence and lack of overfitting.

In [ ]:
# Display training curves generated from training run
curves_path = 'assets/training_curves.png'
if os.path.exists(curves_path):
    from IPython.display import Image, display
    print("Loading saved training curves:")
    display(Image(filename=curves_path))
else:
    print("Training curves plot not found. Run python train_pipeline.py first.")

# Phase 7: Evaluation
We evaluate the trained model on the independent testing set. We calculate and print all 5 required verification metrics (Overall Accuracy, F1 Score, Per-Class Accuracy, EER, and Confusion Matrix Heatmap) and plot the ROC curve.

In [ ]:
from utils.metrics import calculate_metrics
from sklearn.metrics import roc_curve, auc

x_test_path = 'features/X_test.npy'
y_test_path = 'features/y_test.npy'
model_path = 'model/voiceguard_cnn_lstm.h5'

if os.path.exists(x_test_path) and os.path.exists(model_path):
    # Load model and test features
    model = tf.keras.models.load_model(model_path)
    X_test = np.load(x_test_path)
    y_test = np.load(y_test_path)
    
    # Predict probabilities
    print("Running predictions on test set...")
    y_prob = model.predict(X_test, verbose=1).flatten()
    
    # Compute metrics and display report
    metrics_report = calculate_metrics(y_test, y_prob)
    
    # Plot ROC Curve
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    
    fnr = 1 - tpr
    idx = np.argmin(np.abs(fnr - fpr))
    eer = fpr[idx]
    opt_thresh = thresholds[idx]
    
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, color='#00C9A7', lw=2.5, label=f'ROC Curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='#ff7b72', linestyle='--')
    plt.scatter(eer, 1 - eer, color='white', zorder=5, s=60, 
                label=f'EER = {eer*100:.2f}% at Thresh = {opt_thresh:.3f}')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (FPR)')
    plt.ylabel('True Positive Rate (TPR)')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.1)
    plt.show()
else:
    print("Model or test features not found. Train the model first.")

# Phase 8: Inference Demo
Here, we demonstrate running the end-to-end inference pipeline using `predict_single` on a subset of 5 real and 5 fake files from the test set to display model predictions and verification outcomes.

In [ ]:
from utils.predict import predict_single

test_dir = 'data/for-2seconds/testing'
real_dir = os.path.join(test_dir, 'real')
fake_dir = os.path.join(test_dir, 'fake')

if os.path.exists(real_dir) and os.path.exists(fake_dir):
    # Select 5 real files and 5 fake files
    real_files = [os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.lower().endswith(('.wav', '.mp3', '.flac'))][:5]
    fake_files = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if f.lower().endswith(('.wav', '.mp3', '.flac'))][:5]
    
    demo_files = real_files + fake_files
    
    print("=" * 80)
    print(f"{'VOICEGUARD SINGLE-FILE INFERENCE DEMO':^80}")
    print("=" * 80)
    
    for i, file_path in enumerate(demo_files):
        true_label = 'Genuine' if i < 5 else 'Deepfake'
        try:
            res = predict_single(file_path)
            pred_label = 'Genuine' if 'Genuine' in res['label'] else 'Deepfake'
            status = "✅ CORRECT" if true_label == pred_label else "❌ WRONG"
            
            print(f"File   : {os.path.basename(file_path):<30}")
            print(f"True   : {true_label:<10} | Predicted: {pred_label:<10} | Status: {status}")
            print(f"Conf   : {res['confidence']:.2f}% | Prob: {res['fake_probability']:.4f} (Thresh: {res['threshold_used']:.4f})")
            print("-" * 80)
        except Exception as e:
            print(f"File   : {os.path.basename(file_path):<30} | Error during prediction: {e}")
            print("-" * 80)
else:
    print("Test directories not found. Make sure dataset is in data/for-2seconds/")